comfyui_colab_with_manager.ipynb のファイルに対して、利便性を上げるための変更を加えています。

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  # ここをFalseにするだけで、100GBエリアへのインストールに切り替わります
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
!pip3 install accelerate
!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip3 install torchsde
!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece
!pip install comfyui-workflow-templates
!pip install comfyui-embedded-docs

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


In [ ]:
# 画像保存先をGoogleドライブにリンクさせる設定
from google.colab import drive
import os

# 1. Googleドライブをマウント（接続）する
drive.mount('/content/drive')

# 2. ドライブ側に保存用フォルダを作る
drive_output_path = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(drive_output_path, exist_ok=True)

# 3. ComfyUIの出力フォルダをドライブにリンク（シンボリックリンク）する
local_output_path = "/content/ComfyUI/output"

# すでにフォルダがある場合は削除して、リンクに置き換える
if os.path.exists(local_output_path):
    !rm -rf {local_output_path}

!ln -s {drive_output_path} {local_output_path}

print(f"✅ 設定完了: 画像は自動的に {drive_output_path} に保存されます")

Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [ ]:
# Checkpoints

### SDXL
### I recommend these workflow examples: https://comfyanonymous.github.io/ComfyUI_examples/sdxl/

#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors -P ./models/checkpoints/

# SDXL ReVision
#!wget -c https://huggingface.co/comfyanonymous/clip_vision_g/resolve/main/clip_vision_g.safetensors -P ./models/clip_vision/

# SD1.5
# !wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# SD2
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors -P ./models/checkpoints/

# Some SD1.5 anime style
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1_orangemixs.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A3_orangemixs.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/Linaqruf/anything-v3.0/resolve/main/anything-v3-fp16-pruned.safetensors -P ./models/checkpoints/

# Waifu Diffusion 1.5 (anime style SD2.x 768-v)
#!wget -c https://huggingface.co/waifu-diffusion/wd-1-5-beta3/resolve/main/wd-illusion-fp16.safetensors -P ./models/checkpoints/


# unCLIP models
#!wget -c https://huggingface.co/comfyanonymous/illuminatiDiffusionV1_v11_unCLIP/resolve/main/illuminatiDiffusionV1_v11-unclip-h-fp16.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/comfyanonymous/wd-1.5-beta2_unCLIP/resolve/main/wd-1-5-beta2-aesthetic-unclip-h-fp16.safetensors -P ./models/checkpoints/


# VAE
# !wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt -P ./models/vae/
#!wget -c https://huggingface.co/hakurei/waifu-diffusion-v1-4/resolve/main/vae/kl-f8-anime2.ckpt -P ./models/vae/


# Loras
#!wget -c https://civitai.com/api/download/models/10350 -O ./models/loras/theovercomer8sContrastFix_sd21768.safetensors #theovercomer8sContrastFix SD2.x 768-v
#!wget -c https://civitai.com/api/download/models/10638 -O ./models/loras/theovercomer8sContrastFix_sd15.safetensors #theovercomer8sContrastFix SD1.x
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors -P ./models/loras/ #SDXL offset noise lora


# T2I-Adapter
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_depth_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_seg_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_sketch_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_keypose_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_openpose_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_color_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_canny_sd14v1.pth -P ./models/controlnet/

# T2I Styles Model
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_style_sd14v1.pth -P ./models/style_models/

# CLIPVision model (needed for styles model)
#!wget -c https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/pytorch_model.bin -O ./models/clip_vision/clip_vit14.bin


# ControlNet
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_ip2p_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_shuffle_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11f1p_sd15_depth_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_lineart_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_normalbae_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_scribble_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_seg_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15s2_lineart_anime_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11u_sd15_tile_fp16.safetensors -P ./models/controlnet/

# ControlNet SDXL
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors -P ./models/controlnet/

# Controlnet Preprocessor nodes by Fannovel16
#!cd custom_nodes && git clone https://github.com/Fannovel16/comfy_controlnet_preprocessors; cd comfy_controlnet_preprocessors && python install.py


# GLIGEN
#!wget -c https://huggingface.co/comfyanonymous/GLIGEN_pruned_safetensors/resolve/main/gligen_sd14_textbox_pruned_fp16.safetensors -P ./models/gligen/


# ESRGAN upscale model
#!wget -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./models/upscale_models/
#!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth -P ./models/upscale_models/
#!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth -P ./models/upscale_models/




In [ ]:
# # 1. 保存先フォルダを強制的に作成する
# import os
# path = "/content/drive/MyDrive/ComfyUI/models/checkpoints"
# # os.makedirs(path, exist_ok=True)

# # 2. そのフォルダへ移動する
# %cd {path}

# # 3. 再度ダウンロードを実行する
# !wget -c -O ace_step_v1_3.5b.safetensors "https://huggingface.co/Comfy-Org/ACE-Step_ComfyUI_repackaged/resolve/main/all_in_one/ace_step_v1_3.5b.safetensors?download=true"

# path = "/content/drive/MyDrive"
# # os.makedirs(path, exist_ok=True)
# %cd {path}


In [ ]:
# # 1. 100GBエリア（/content/）のモデルフォルダへ移動
# # ※ USE_GOOGLE_DRIVE=Falseにしたので、ComfyUIは /content/ComfyUI にあります
# %cd /content/ComfyUI/models/checkpoints/

# # 2. ACE-Stepモデルをダウンロード（7.2GBあっても100GBエリアなら余裕です）
# !wget -c -O ace_step_v1_3.5b.safetensors "https://huggingface.co/Comfy-Org/ACE-Step_ComfyUI_repackaged/resolve/main/all_in_one/ace_step_v1_3.5b.safetensors?download=true"

# print("✅ ダウンロード完了（このファイルはブラウザを閉じると消えますが、ドライブ容量は消費しません）")

In [ ]:
# Hunyuan3D-V2 (Multi-View) モデルのダウンロード
# r1の設定に合わせ、100GBエリアのチェックポイントフォルダへ保存します
%cd /content/ComfyUI/models/checkpoints/

!wget -c "https://huggingface.co/Comfy-Org/hunyuan3D_2.0_repackaged/resolve/main/split_files/hunyuan3d-dit-v2-mv-turbo_fp16.safetensors"

print("✅ Hunyuan3D-V2 のダウンロードが完了しました")

---
## 🎨 Hunyuan3D テクスチャ生成セットアップ（T4対応版）

以下のセルを **順番に** 実行してください。形状生成＋テクスチャ生成の完全パイプラインが使えるようになります。

| モデル | 必要VRAM |
|--------|----------|
| 形状生成のみ（mini） | 5 GB |
| 形状生成（標準） | 6 GB |
| 形状＋テクスチャ（完全） | **12 GB** ✅ T4対応 |


In [ ]:
# ============================================================
# Step 1: ComfyUI-Hunyuan3DWrapper のインストール
# ============================================================
import os

wrapper_dir = "/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper"
%cd /content/ComfyUI/custom_nodes

if not os.path.exists(wrapper_dir):
    !git clone https://github.com/kijai/ComfyUI-Hunyuan3DWrapper
    print("✅ ComfyUI-Hunyuan3DWrapper をクローンしました")
else:
    %cd ComfyUI-Hunyuan3DWrapper
    !git pull
    %cd ..
    print("✅ ComfyUI-Hunyuan3DWrapper を更新しました")

%cd /content/ComfyUI

In [ ]:
# ============================================================
# Step 2: 依存パッケージのインストール
# ============================================================
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/requirements.txt
!pip install -q trimesh xatlas rembg huggingface_hub
print("✅ 依存パッケージのインストール完了")

In [ ]:
# ============================================================
# Step 3: custom_rasterizer のビルド（テクスチャ生成に必須）
# ============================================================
import subprocess, sys, os

wrapper = "/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper"
rast_dir = f"{wrapper}/hy3dgen/texgen/custom_rasterizer"

# --- 事前確認 ---
print("=== ビルド前確認 ===")
!python3 -c "import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)"
!nvcc --version 2>&1 | head -4

# --- ビルドに必要なパッケージ ---
!pip install -q ninja wheel setuptools

# --- setup.py の内容確認 ---
setup_py = os.path.join(rast_dir, 'setup.py')
if os.path.exists(setup_py):
    print("\n📄 setup.py 確認:")
    !head -30 {setup_py}
else:
    print(f"❌ setup.py が見つかりません: {setup_py}")

# --- ビルド実行（3段階フォールバック）---
print("\n🔨 custom_rasterizer をビルド中...")

# 方法1: pip install -e . --no-build-isolation
r1 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".",
     "--no-build-isolation", "--verbose"],
    cwd=rast_dir, capture_output=True, text=True
)
if r1.returncode == 0:
    print("✅ 方法1 成功: pip install -e .")
else:
    print("⚠️ 方法1 失敗、方法2を試みます...")
    print(r1.stderr[-1000:])

    # 方法2: python setup.py build_ext --inplace
    r2 = subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=rast_dir, capture_output=True, text=True
    )
    if r2.returncode == 0:
        print("✅ 方法2 成功: setup.py build_ext")
    else:
        print("⚠️ 方法2 失敗、方法3を試みます...")
        print(r2.stderr[-1000:])

        # 方法3: MAX_JOBSを1にしてシングルスレッドビルド
        env = os.environ.copy()
        env['MAX_JOBS'] = '1'
        env['TORCH_CUDA_ARCH_LIST'] = '7.5'  # T4のアーキテクチャ
        r3 = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", ".",
             "--no-build-isolation"],
            cwd=rast_dir, capture_output=True, text=True, env=env
        )
        if r3.returncode == 0:
            print("✅ 方法3 成功（T4最適化ビルド）")
        else:
            print("❌ 全てのビルド方法が失敗しました")
            print("最終エラー:")
            print(r3.stderr[-2000:])

# --- インポートテスト ---
print("\n=== インポートテスト ===")
test = subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0, '/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/hy3dgen/texgen/custom_rasterizer'); import custom_rasterizer; print('✅ custom_rasterizer インポート成功:', custom_rasterizer.__file__)"],
    capture_output=True, text=True
)
if test.returncode == 0:
    print(test.stdout)
else:
    # .so ファイルを直接探す
    print("⚠️ 標準インポート失敗。.soファイルを検索中...")
    find_result = subprocess.run(
        ['find', rast_dir, '-name', '*.so', '-o', '-name', '*.pyd'],
        capture_output=True, text=True
    )
    so_files = find_result.stdout.strip()
    if so_files:
        print("見つかった .so/.pyd ファイル:")
        print(so_files)
        # .soのディレクトリをsys.pathに追加してリトライ
        so_dir = os.path.dirname(so_files.split('\n')[0])
        test2 = subprocess.run(
            [sys.executable, "-c",
             f"import sys; sys.path.insert(0, '{so_dir}'); import custom_rasterizer; print('✅ インポート成功 (パス指定):', custom_rasterizer.__file__)"],
            capture_output=True, text=True
        )
        if test2.returncode == 0:
            print(test2.stdout)
            print(f"\n⚠️ ComfyUI起動時に以下を追加してください:")
            print(f"   sys.path.insert(0, '{so_dir}')")
        else:
            print("❌ インポートできませんでした")
            print(test2.stderr)
    else:
        print("❌ .so/.pydファイルが見つかりません。ビルドが失敗しています")
        print(test.stderr[-500:])


In [ ]:
# ============================================================
# Step 4: モデルのダウンロード（T4対応・12GB以内）
# ============================================================
import os

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/diffusers", exist_ok=True)

# --- 形状生成モデル（mv-turbo）---
shape_model_path = "/content/ComfyUI/models/checkpoints/hunyuan3d-dit-v2-mv-turbo_fp16.safetensors"
if not os.path.exists(shape_model_path):
    print("📥 形状生成モデル (mv-turbo) をダウンロード中...")
    !wget -q --show-progress -c \
        "https://huggingface.co/Comfy-Org/hunyuan3D_2.0_repackaged/resolve/main/split_files/hunyuan3d-dit-v2-mv-turbo_fp16.safetensors" \
        -P /content/ComfyUI/models/checkpoints/
    print("✅ 形状生成モデル完了")
else:
    print("✅ 形状生成モデルはすでにあります")

# --- テクスチャ生成モデル（hunyuan3d-paint-v2-0-turbo）---
# ⚠️ turboのVAEはbin形式のみのため、hf_transferで確実にDLし、必要なら変換する
paint_dir = "/content/ComfyUI/models/diffusers/hunyuan3d-paint-v2-0-turbo"

print("\n📥 テクスチャ生成モデル (paint-v2-0-turbo) をダウンロード中...")
!pip install -q huggingface_hub hf_transfer

# hf_transferを使って高速DL
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="tencent/Hunyuan3D-2",
    allow_patterns=["hunyuan3d-paint-v2-0-turbo/**"],
    local_dir="/content/ComfyUI/models/diffusers",
    ignore_patterns=["*.md", "*.txt"]
)
print("✅ テクスチャモデルのダウンロード完了")

# --- .bin → .safetensors 変換（必要な場合のみ）---
print("\n🔄 .bin → .safetensors 変換チェック中...")
!pip install -q safetensors torch

import torch
from safetensors.torch import save_file

def convert_bin_to_safetensors(directory):
    for root, dirs, files in os.walk(directory):
        for fname in files:
            if fname == "diffusion_pytorch_model.bin":
                bin_path = os.path.join(root, fname)
                st_path = os.path.join(root, "diffusion_pytorch_model.safetensors")
                if not os.path.exists(st_path):
                    print(f"  変換中: {bin_path}")
                    weights = torch.load(bin_path, map_location="cpu")
                    # state_dictが入れ子の場合
                    if "state_dict" in weights:
                        weights = weights["state_dict"]
                    # float32のみsafetensorsに保存可能（float16はそのまま）
                    save_file(weights, st_path)
                    print(f"  ✅ 変換完了: {st_path}")
                else:
                    print(f"  ✅ すでに存在: {st_path}")

convert_bin_to_safetensors("/content/ComfyUI/models/diffusers/hunyuan3d-paint-v2-0-turbo")

# --- RealESRGAN ---
realesrgan_dir = "/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/ckpt"
os.makedirs(realesrgan_dir, exist_ok=True)
realesrgan_path = os.path.join(realesrgan_dir, "RealESRGAN_x4plus.pth")
if not os.path.exists(realesrgan_path):
    print("\n📥 RealESRGAN をダウンロード中...")
    !wget -q --show-progress \
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" \
        -P {realesrgan_dir}
else:
    print("\n✅ RealESRGAN はすでにあります")

print("\n✅ 全モデルのダウンロード・変換が完了しました")

In [ ]:
# ============================================================
# Step 4.5: tokenizer の LFS ファイルを直接ダウンロードして上書き
# （snapshot_download では vocab.json / merges.txt が空になる問題を修正）
# ============================================================
import os

tokenizer_dir = "/content/ComfyUI/models/diffusers/hunyuan3d-paint-v2-0-turbo/tokenizer"
os.makedirs(tokenizer_dir, exist_ok=True)

BASE = "https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-paint-v2-0-turbo/tokenizer"

lfs_files = [
    "vocab.json",
    "merges.txt",
    "special_tokens_map.json",
    "tokenizer_config.json",
]

print("📥 tokenizer ファイルを直接ダウンロード中...")
for fname in lfs_files:
    out_path = os.path.join(tokenizer_dir, fname)
    !wget -q --show-progress -c "{BASE}/{fname}" -O "{out_path}"

# ファイルサイズ確認
print("\n=== tokenizer ファイル確認 ===")
for fname in lfs_files:
    path = os.path.join(tokenizer_dir, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    status = "✅" if size > 100 else "❌ 空または不正"
    print(f"  {status} {fname}: {size:,} bytes")

# text_encoder の tokenizer も同様に確認・修正
te_tokenizer_dir = "/content/ComfyUI/models/diffusers/hunyuan3d-paint-v2-0-turbo/text_encoder"
if os.path.exists(te_tokenizer_dir):
    te_base = "https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-paint-v2-0-turbo/text_encoder"
    # text_encoderにtokenizerフォルダがある場合も同様に対処
    for root, dirs, files in os.walk(te_tokenizer_dir):
        for f in files:
            fpath = os.path.join(root, f)
            if os.path.getsize(fpath) < 100 and f.endswith(('.json', '.txt')):
                rel = os.path.relpath(fpath, "/content/ComfyUI/models/diffusers/hunyuan3d-paint-v2-0-turbo")
                url = f"https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-paint-v2-0-turbo/{rel}"
                print(f"  修正中: {rel}")
                !wget -q -c "{url}" -O "{fpath}"

print("\n✅ tokenizer ファイルの修正完了")

In [ ]:
# ============================================================
# Step 5: インストール確認（テクスチャ生成の動作チェック）
# ============================================================
import os, subprocess, sys

wrapper = "/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper"
paint_dir = "/content/ComfyUI/models/diffusers/hunyuan3d-paint-v2-0-turbo"

checks = [
    ("ComfyUI-Hunyuan3DWrapper",       wrapper),
    ("形状モデル (mv-turbo)",
     "/content/ComfyUI/models/checkpoints/hunyuan3d-dit-v2-mv-turbo_fp16.safetensors"),
    ("テクスチャモデル (paint-v2-0-turbo)", paint_dir),
    ("RealESRGAN",
     f"{wrapper}/ckpt/RealESRGAN_x4plus.pth"),
]

print("=" * 45)
print("  インストール確認")
print("=" * 45)

all_ok = True
for name, path in checks:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {name}")
    if not exists:
        all_ok = False

# tokenizer ファイルサイズ確認（LFS問題チェック）
print()
print("--- tokenizer ファイルサイズ ---")
for fname in ["vocab.json", "merges.txt"]:
    p = os.path.join(paint_dir, "tokenizer", fname)
    size = os.path.getsize(p) if os.path.exists(p) else 0
    ok = size > 10000
    print(f"{'✅' if ok else '❌'} tokenizer/{fname}: {size:,} bytes")
    if not ok:
        all_ok = False

# custom_rasterizer インポート確認
print()
print("--- C++ コンポーネント ---")
rast = subprocess.run([sys.executable, "-c", "import custom_rasterizer"],
                      capture_output=True, text=True)
rast_ok = rast.returncode == 0
print(f"{'✅' if rast_ok else '❌'} custom_rasterizer (テクスチャ生成の核心)")
if not rast_ok:
    all_ok = False
    print("  → Step 3 を再実行してください")

# diffusion_pytorch_model.safetensors 存在確認
print()
print("--- モデルウェイトファイル ---")
for subdir in ["unet", "vae", ""]:
    d = os.path.join(paint_dir, subdir) if subdir else paint_dir
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith((".safetensors", ".bin")) and not os.path.isdir(os.path.join(d, f)):
                size = os.path.getsize(os.path.join(d, f))
                label = subdir if subdir else "root"
                ok = size > 1_000_000
                print(f"  {'✅' if ok else '❌'} {label}/{f}: {size/1024/1024:.1f} MB")

print()
if all_ok and rast_ok:
    print("🎉 すべてOK！ComfyUI を起動してワークフローを実行してください。")
else:
    print("⚠️ 問題があります。❌ の項目を確認して対応するStepを再実行してください。")


In [ ]:
!pip install av
!pip install comfy_aimdo

### Run ComfyUI with cloudflared (Recommended Way)




In [ ]:
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
# %cd /content/drive/MyDrive/ComfyUI
# !python main.py --dont-print-server

%cd /content/ComfyUI

# CPUで動かす場合はこちら（GPU制限解除待ちの場合）
# !python main.py --cpu --dont-print-server

# もしT4 GPUが使えるようになったら、--cpu を外して以下にしてください
!python main.py --dont-print-server

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server